# 🏥 GI Tract Anatomical Region Detection — GALAR Dataset
## ICPR 2026 RARE-VISION Competition | Research-Grade Solution
### Multi-Label Classification: Detect Which GI Region Appears in Each Capsule Frame

**Targets (9 anatomy + 1 transition):** `z-line`, `pylorus`, `ampulla of vater`,
`ileocecal valve`, `section`, `mouth`, `esophagus`, `stomach`, `small intestine`, `colon`

**Pipeline:**
- Patient-level stratified split (no data leakage)
- EfficientNet-B4 NoisyStudent + GeM Pooling + auxiliary disease head
- Focal Loss + Label Smoothing + pos_weight for class imbalance
- MixUp / CutMix adapted for multi-label targets
- Layer-wise LR decay + Cosine Annealing Warm Restarts
- 5-view Test-Time Augmentation (TTA)
- Per-class threshold optimization
- Temporal smoothing for video-level inference
- GradCAM++ explainability

**References:** EfficientNet (Tan & Le, ICML 2019) · Focal Loss (Lin et al., ICCV 2017) ·
MixUp (Zhang et al., ICLR 2018) · CutMix (Yun et al., ICCV 2019) ·
GradCAM++ (Chattopadhyay et al., WACV 2018) · GeM Pooling (Radenovic et al., TPAMI 2019)


## 📦 Section 1: Install Dependencies

In [ ]:
!pip install timm -q
!pip install grad-cam -q
!pip install albumentations -q
!pip install seaborn -q


## 📚 Section 2: Imports & Reproducibility

In [ ]:
import os, json, random, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, roc_curve, auc
)

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device  : {device}")
print(f"✅ PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU     : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## ⚙️ Section 3: Configuration

In [ ]:
class CFG:
    # ── Paths (auto-detected below) ───────────────────────────
    BASE_DIR   = "/kaggle/input"           # Kaggle input root
    OUTPUT_DIR = "/kaggle/working"
    MODEL_PATH = "/kaggle/working/best_anatomy_model.pth"

    # ── All anatomy region columns in the CSV ─────────────────
    ANATOMY_COLS = [
        'z-line', 'pylorus', 'ampulla of vater',
        'ileocecal valve', 'section', 'mouth',
        'esophagus', 'stomach', 'small intestine', 'colon'
    ]

    # ── Disease columns (used for auxiliary training task) ────
    DISEASE_COLS = [
        'ulcer', 'polyp', 'active bleeding', 'blood',
        'erythema', 'erosion', 'angiectasia', 'IBD',
        'foreign body', 'esophagitis', 'varices',
        'hematin', 'celiac', 'cancer', 'lymphangioectasis'
    ]

    NUM_ANATOMY = len(ANATOMY_COLS)   # 10
    NUM_DISEASE = len(DISEASE_COLS)   # 15

    # ── Image ──────────────────────────────────────────────────
    IMG_SIZE    = 224
    NUM_WORKERS = 2
    PIN_MEMORY  = True

    # ── Model ──────────────────────────────────────────────────
    MODEL_NAME  = "tf_efficientnet_b4_ns"   # NoisyStudent
    PRETRAINED  = True
    DROP_RATE   = 0.3
    DROP_PATH   = 0.2

    # ── Training ───────────────────────────────────────────────
    EPOCHS       = 25
    BATCH_SIZE   = 32
    LR           = 3e-4
    MIN_LR       = 1e-6
    WEIGHT_DECAY = 1e-4
    PATIENCE     = 6

    # ── Loss ───────────────────────────────────────────────────
    FOCAL_GAMMA  = 2.0
    LABEL_SMOOTH = 0.05
    AUX_WEIGHT   = 0.3    # weight of disease auxiliary loss

    # ── Augmentation ───────────────────────────────────────────
    MIXUP_ALPHA  = 0.3
    CUTMIX_ALPHA = 1.0
    MIXUP_PROB   = 0.4

    # ── Inference ──────────────────────────────────────────────
    DEFAULT_THRESHOLD  = 0.5
    TTA_STEPS          = 5
    TEMPORAL_WINDOW    = 7    # frames for rolling average smoothing

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print(f"✅ Config ready  |  {CFG.NUM_ANATOMY} anatomy targets  |  {CFG.NUM_DISEASE} disease targets")


## 📂 Section 4: Auto-Detect Dataset Path

In [ ]:
def find_dataset_root(base="/kaggle/input"):
    """
    Walks /kaggle/input to find the folder that contains
    both a 'CSVs' subfolder and an 'Images' subfolder.
    Works regardless of the Kaggle dataset slug name.
    """
    for root, dirs, files in os.walk(base):
        subdirs = set(os.listdir(root))
        if 'CSVs' in subdirs and 'Images' in subdirs:
            print(f"✅ Dataset root found: {root}")
            return root
        # Stop going deeper than 3 levels
        depth = root.replace(base, '').count(os.sep)
        if depth >= 3:
            dirs.clear()

    raise FileNotFoundError(
        "❌ Could not find a folder with 'CSVs' and 'Images' under /kaggle/input\n"
        "   Make sure 'Rare_Events_Dataset' is added to your notebook."
    )

DATA_DIR = find_dataset_root(CFG.BASE_DIR)
CSV_DIR  = os.path.join(DATA_DIR, "CSVs")
IMG_DIR  = os.path.join(DATA_DIR, "Images")

print(f"   CSV_DIR  : {CSV_DIR}")
print(f"   IMG_DIR  : {IMG_DIR}")
print(f"   CSVs     : {sorted(os.listdir(CSV_DIR))}")
print(f"   Patients : {sorted(os.listdir(IMG_DIR))}")


## 📋 Section 5: Load & Parse All Patient CSVs

In [ ]:
def load_all_csvs(csv_dir, img_dir):
    """
    Reads every patient CSV and assembles a master dataframe.

    Each row represents ONE video frame and contains:
      patient_id  – patient folder number (e.g. '61')
      frame       – frame number (integer)
      img_path    – full path to the PNG file
      <anatomy>   – binary label for each of 10 anatomy columns
      <disease>   – binary label for each of 15 disease columns

    Only rows whose image file actually exists on disk are kept.
    """
    all_dfs        = []
    csv_files      = sorted([f for f in os.listdir(csv_dir) if f.endswith('.csv')])
    missing_images = 0

    print(f"Found {len(csv_files)} patient CSV(s): {csv_files}")

    for csv_file in tqdm(csv_files, desc="Loading CSVs"):
        patient_id = os.path.splitext(csv_file)[0]
        df = pd.read_csv(os.path.join(csv_dir, csv_file))

        df['patient_id'] = patient_id

        # Build image path from the 'frame' column
        # Frame 15480  →  frame_015480.PNG
        def make_path(f):
            try:
                return os.path.join(img_dir, patient_id, f"frame_{int(f):06d}.PNG")
            except Exception:
                return ""

        df['img_path'] = df['frame'].apply(make_path)

        # Drop rows where image file is missing
        exists_mask     = df['img_path'].apply(os.path.exists)
        missing_images += (~exists_mask).sum()
        df = df[exists_mask].reset_index(drop=True)

        # Fill NaN labels with 0 and cast to int
        all_label_cols = CFG.ANATOMY_COLS + CFG.DISEASE_COLS
        for col in all_label_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            else:
                df[col] = 0

        all_dfs.append(df)

    master = pd.concat(all_dfs, ignore_index=True)
    print(f"\n{'='*50}")
    print(f"✅ Total frames  : {len(master):,}")
    print(f"   Missing imgs  : {missing_images:,}")
    print(f"   Patients      : {master['patient_id'].nunique()}")
    print(f"   Columns       : {list(master.columns)}")
    return master


master_df = load_all_csvs(CSV_DIR, IMG_DIR)


## 📊 Section 6: Exploratory Data Analysis (EDA)

In [ ]:
# ── Anatomy label distribution ────────────────────────────────
anatomy_counts = master_df[CFG.ANATOMY_COLS].sum().sort_values(ascending=False)
disease_counts = master_df[CFG.DISEASE_COLS].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

colors_a = sns.color_palette("Blues_d", len(anatomy_counts))
bars = axes[0].bar(anatomy_counts.index, anatomy_counts.values, color=colors_a, edgecolor='k', lw=0.5)
axes[0].set_title('📍 Anatomical Region Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Frame Count'); axes[0].tick_params(axis='x', rotation=40)
for b, v in zip(bars, anatomy_counts.values):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+30, f'{v:,}',
                 ha='center', fontsize=8, fontweight='bold')

colors_d = sns.color_palette("Reds_d", len(disease_counts))
bars2 = axes[1].bar(disease_counts.index, disease_counts.values, color=colors_d, edgecolor='k', lw=0.5)
axes[1].set_title('🦠 Disease Label Distribution (auxiliary)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Frame Count'); axes[1].tick_params(axis='x', rotation=40)
for b, v in zip(bars2, disease_counts.values):
    if v > 0:
        axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+5, f'{v:,}',
                     ha='center', fontsize=7)

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Anatomy class summary:")
print(f"  {'Region':25s}  {'Count':>8}  {'% frames':>9}  {'Imbalance ratio':>16}")
max_c = anatomy_counts.max()
for col, cnt in anatomy_counts.items():
    pct   = cnt / len(master_df) * 100
    ratio = max_c / (cnt + 1)
    bar   = '█' * max(1, int(pct/2))
    print(f"  {col:25s}  {cnt:>8,}  {pct:>8.1f}%  {ratio:>14.1f}x  {bar}")


In [ ]:
# ── Labels per frame analysis ─────────────────────────────────
labels_per_frame = master_df[CFG.ANATOMY_COLS].sum(axis=1)
print("Labels per frame distribution:")
for n, cnt in sorted(Counter(labels_per_frame).items()):
    pct = cnt / len(master_df) * 100
    print(f"  {n:2d} label(s) : {cnt:6,} frames ({pct:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(labels_per_frame, bins=range(0, int(labels_per_frame.max())+2),
             color='steelblue', edgecolor='k', alpha=0.85)
axes[0].set_title('Labels per Frame', fontweight='bold')
axes[0].set_xlabel('# anatomy labels'); axes[0].set_ylabel('Frame count')
axes[0].grid(axis='y', alpha=0.3)

# Per-patient heatmap
patient_cov = master_df.groupby('patient_id')[CFG.ANATOMY_COLS].sum()
sns.heatmap(patient_cov, cmap='YlOrRd', annot=True, fmt='d',
            linewidths=0.5, ax=axes[1], cbar_kws={'label':'frames'})
axes[1].set_title('Per-Patient Anatomy Coverage', fontweight='bold')
axes[1].tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/eda_detail.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Sample image per anatomy class ───────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for i, region in enumerate(CFG.ANATOMY_COLS):
    rows = master_df[master_df[region] == 1]
    ax   = axes[i]
    if len(rows) == 0:
        ax.set_title(f'{region}\n(no samples)', fontsize=9); ax.axis('off'); continue
    row = rows.sample(1, random_state=SEED).iloc[0]
    try:
        img     = Image.open(row['img_path']).convert('RGB')
        active  = [c for c in CFG.ANATOMY_COLS if row[c] == 1]
        diseases = [c for c in CFG.DISEASE_COLS if row[c] == 1]
        ax.imshow(img)
        ax.set_title(f'{region}\nPatient {row.patient_id}  frame {int(row.frame)}\n'
                     f'All: {active}', fontsize=7, fontweight='bold')
        if diseases:
            ax.set_xlabel(f'Disease: {diseases}', fontsize=6, color='red')
    except Exception as e:
        ax.set_title(f'{region}\n(error)', fontsize=9)
    ax.axis('off')

plt.suptitle('🔬 Sample Frame per Anatomy Region', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/sample_frames.png', dpi=150, bbox_inches='tight')
plt.show()


## ✂️ Section 7: Patient-Level Train / Val / Test Split

In [ ]:
# ── CRITICAL: split by patient, NOT by frame ─────────────────
# Mixing frames from the same patient across train/val would be
# data leakage — the model would learn patient-specific image
# artifacts (lighting, camera orientation) rather than anatomy.

patients = sorted(master_df['patient_id'].unique())
print(f"Total patients: {len(patients)}  →  {patients}")

train_pids, temp_pids = train_test_split(
    patients, test_size=0.30, random_state=SEED)
val_pids, test_pids   = train_test_split(
    temp_pids, test_size=0.50, random_state=SEED)

train_df = master_df[master_df['patient_id'].isin(train_pids)].reset_index(drop=True)
val_df   = master_df[master_df['patient_id'].isin(val_pids)].reset_index(drop=True)
test_df  = master_df[master_df['patient_id'].isin(test_pids)].reset_index(drop=True)

print(f"\nTrain patients : {sorted(train_pids)}")
print(f"Val   patients : {sorted(val_pids)}")
print(f"Test  patients : {sorted(test_pids)}")
print(f"\nTrain frames   : {len(train_df):,}")
print(f"Val   frames   : {len(val_df):,}")
print(f"Test  frames   : {len(test_df):,}")

# Verify no overlap
assert not set(train_pids) & set(val_pids)
assert not set(train_pids) & set(test_pids)
print("\n✅ No patient leakage between splits")


## 🎨 Section 8: Augmentation Pipeline

In [ ]:
def get_train_transforms(img_size=CFG.IMG_SIZE):
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.75, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.12,
                           rotate_limit=20, p=0.4),
        # Endoscopy-specific color augmentation
        A.OneOf([
            A.ColorJitter(brightness=0.3, contrast=0.3,
                          saturation=0.2, hue=0.1, p=1.0),
            A.CLAHE(clip_limit=3.0, p=1.0),
            A.HueSaturationValue(hue_shift_limit=15,
                                 sat_shift_limit=25,
                                 val_shift_limit=20, p=1.0),
        ], p=0.6),
        # Noise/blur — simulates VCE motion and compression artifacts
        A.OneOf([
            A.GaussNoise(var_limit=(5, 30), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
        ], p=0.25),
        A.OneOf([
            A.GridDistortion(p=1.0),
            A.ElasticTransform(alpha=1, sigma=40, p=1.0),
        ], p=0.2),
        A.CoarseDropout(max_holes=6, max_height=img_size//10,
                        max_width=img_size//10, fill_value=0, p=0.25),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_val_transforms(img_size=CFG.IMG_SIZE):
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_tta_transforms(img_size=CFG.IMG_SIZE):
    """All 5 TTA variants — Resize MUST be first to handle variable input sizes"""
    norm = [A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()]
    return [
        A.Compose([A.Resize(height=img_size, width=img_size)] + norm),
        A.Compose([A.Resize(height=img_size, width=img_size), A.HorizontalFlip(p=1.0)] + norm),
        A.Compose([A.Resize(height=img_size, width=img_size), A.VerticalFlip(p=1.0)] + norm),
        A.Compose([A.Resize(height=img_size, width=img_size), A.Rotate(limit=(90,90), p=1.0)] + norm),
        A.Compose([A.Resize(height=img_size, width=img_size), A.Rotate(limit=(-90,-90), p=1.0)] + norm),
    ]

print("✅ Augmentation pipelines ready")


## 🗃️ Section 9: Dataset & DataLoaders

In [ ]:
class GalarDataset(Dataset):
    """
    Loads GALAR frames on-demand.

    Returns per sample:
      image          : (3, H, W) float tensor
      anatomy_labels : (10,) float tensor — multi-label binary
      disease_labels : (15,) float tensor — for auxiliary task
    """
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = np.array(Image.open(row['img_path']).convert('RGB'))

        anatomy_labels = torch.tensor(
            row[CFG.ANATOMY_COLS].values.astype(np.float32), dtype=torch.float32)
        disease_labels = torch.tensor(
            row[CFG.DISEASE_COLS].values.astype(np.float32), dtype=torch.float32)

        if self.transform:
            img = self.transform(image=img)['image']

        return img, anatomy_labels, disease_labels


# ── Weighted sampler — rare regions seen as often as common ones ──
def get_anatomy_sampler(df):
    """
    Each frame's weight = inverse frequency of its rarest anatomy label.
    Frames with no anatomy label get minimum weight.
    """
    counts  = df[CFG.ANATOMY_COLS].sum(axis=0).values.astype(float) + 1.0
    weights = 1.0 / counts

    sample_weights = []
    for _, row in df[CFG.ANATOMY_COLS].iterrows():
        active = row.values
        if active.sum() == 0:
            sample_weights.append(float(weights.min()))
        else:
            sample_weights.append(float(weights[active == 1].max()))

    return WeightedRandomSampler(
        weights     = torch.tensor(sample_weights, dtype=torch.float),
        num_samples = len(sample_weights),
        replacement = True
    )


train_ds = GalarDataset(train_df, transform=get_train_transforms())
val_ds   = GalarDataset(val_df,   transform=get_val_transforms())
test_ds  = GalarDataset(test_df,  transform=get_val_transforms())

sampler = get_anatomy_sampler(train_df)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                           num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False,
                           num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
test_loader  = DataLoader(test_ds,  batch_size=CFG.BATCH_SIZE, shuffle=False,
                           num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)

print(f"✅ Train batches : {len(train_loader):,}")
print(f"✅ Val   batches : {len(val_loader):,}")
print(f"✅ Test  batches : {len(test_loader):,}")

# Quick sanity check
imgs, anatomy_lbl, disease_lbl = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  images         : {imgs.shape}")
print(f"  anatomy labels : {anatomy_lbl.shape}")
print(f"  disease labels : {disease_lbl.shape}")


## 🧠 Section 10: Model Architecture

In [ ]:
class GeM(nn.Module):
    """
    Generalized Mean Pooling — Radenovic et al., TPAMI 2019.
    Learns the optimal pooling exponent p for the task.
    p=1 → average pool, p→∞ → max pool.
    """
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)


class AnatomyDetector(nn.Module):
    """
    Multi-label GI anatomy region detector.

    Architecture:
      EfficientNet-B4 NoisyStudent (backbone, pretrained)
        → GeM Pooling
        → Shared feature block (512-dim)
        ├── Anatomy Head → 10 logits (primary task)
        └── Disease Head → 15 logits (auxiliary task)

    Auxiliary disease head:
      Anatomy and disease co-occur (e.g. polyps appear in colon,
      ulcers in stomach). Joint training improves shared features.
      Disease loss weighted by CFG.AUX_WEIGHT (default 0.3).
    """
    def __init__(self, num_anatomy=CFG.NUM_ANATOMY,
                 num_disease=CFG.NUM_DISEASE,
                 model_name=CFG.MODEL_NAME,
                 pretrained=CFG.PRETRAINED,
                 drop_rate=CFG.DROP_RATE):
        super().__init__()

        # ── Backbone ──────────────────────────────────────────
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool='',
            drop_rate=drop_rate,
            drop_path_rate=CFG.DROP_PATH
        )
        in_f = self.backbone.num_features

        self.pool = GeM()

        # ── Shared Block ──────────────────────────────────────
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_f, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(drop_rate),
        )

        # ── Anatomy Head (PRIMARY) ────────────────────────────
        self.anatomy_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(drop_rate * 0.5),
            nn.Linear(256, num_anatomy)
        )

        # ── Disease Head (AUXILIARY) ──────────────────────────
        self.disease_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(drop_rate * 0.5),
            nn.Linear(256, num_disease)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        feats  = self.backbone(x)
        pooled = self.pool(feats)
        shared = self.shared(pooled)
        return self.anatomy_head(shared), self.disease_head(shared)

    def predict_proba(self, x):
        """Sigmoid probabilities for anatomy — use for inference"""
        a_logits, _ = self.forward(x)
        return torch.sigmoid(a_logits)


model = AnatomyDetector().to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model     : {CFG.MODEL_NAME}")
print(f"   Total params     : {total_p:,}")
print(f"   Trainable params : {trainable_p:,}")

dummy = torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(device)
a_out, d_out = model(dummy)
print(f"   Anatomy output   : {a_out.shape}  (batch, {CFG.NUM_ANATOMY})")
print(f"   Disease output   : {d_out.shape}  (batch, {CFG.NUM_DISEASE})")


## ⚖️ Section 11: Loss Functions

In [ ]:
class FocalBCELoss(nn.Module):
    """
    Multi-label Focal Loss — Lin et al., ICCV 2017.
    FL = -(1-pt)^gamma * log(pt)
    Down-weights easy examples so training focuses on hard/rare cases.
    """
    def __init__(self, gamma=CFG.FOCAL_GAMMA, pos_weight=None):
        super().__init__()
        self.gamma      = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none')
        pt    = torch.exp(-bce)
        focal = ((1 - pt) ** self.gamma) * bce
        return focal.mean()


class CombinedMultiLabelLoss(nn.Module):
    """60% Focal + 40% BCE with label smoothing"""
    def __init__(self, pos_weight=None, smooth=CFG.LABEL_SMOOTH):
        super().__init__()
        self.focal  = FocalBCELoss(pos_weight=pos_weight)
        self.smooth = smooth

    def forward(self, logits, targets):
        t_smooth   = targets * (1 - self.smooth) + 0.5 * self.smooth
        focal_loss = self.focal(logits, t_smooth)
        bce_loss   = F.binary_cross_entropy_with_logits(logits, t_smooth, reduction='mean')
        return 0.6 * focal_loss + 0.4 * bce_loss


# ── Compute per-class positive weights from training distribution ──
anatomy_pos_counts = train_df[CFG.ANATOMY_COLS].sum(axis=0).values.astype(float)
neg_counts         = len(train_df) - anatomy_pos_counts
pos_weights        = np.clip(neg_counts / (anatomy_pos_counts + 1e-6), 1.0, 20.0)
pos_weight_tensor  = torch.tensor(pos_weights, dtype=torch.float32).to(device)

anatomy_criterion = CombinedMultiLabelLoss(pos_weight=pos_weight_tensor).to(device)
disease_criterion = CombinedMultiLabelLoss().to(device)

print("✅ Loss functions ready")
print("\nAnatomy positive weights (imbalance correction):")
for col, w in zip(CFG.ANATOMY_COLS, pos_weights):
    bar = '█' * min(20, int(w))
    print(f"  {col:25s} {w:5.1f}x  {bar}")


## 📐 Section 12: Optimizer & Scheduler

In [ ]:
def get_optimizer(model, base_lr=CFG.LR, wd=CFG.WEIGHT_DECAY):
    """
    Layer-wise learning rate decay:
      Early backbone layers  → base_lr × 0.01  (preserve pretrained features)
      Middle layers          → base_lr × 0.10
      Late backbone layers   → base_lr × 1.00
      Custom heads / pooling → base_lr × 1.00
    """
    early_p  = []
    middle_p = []
    late_p   = []
    head_p   = []
    other_p  = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(k in name for k in ['anatomy_head','disease_head','shared','pool']):
            head_p.append(param)
        elif any(f'blocks.{i}.' in name for i in [0,1]) or              any(k in name for k in ['conv_stem','bn1']):
            early_p.append(param)
        elif any(f'blocks.{i}.' in name for i in [2,3,4]):
            middle_p.append(param)
        elif any(f'blocks.{i}.' in name for i in [5,6]) or              any(k in name for k in ['conv_head','bn2']):
            late_p.append(param)
        else:
            other_p.append(param)

    groups = []
    if early_p:  groups.append({'params': early_p,  'lr': base_lr * 0.01})
    if middle_p: groups.append({'params': middle_p, 'lr': base_lr * 0.10})
    if late_p:   groups.append({'params': late_p,   'lr': base_lr})
    if head_p:   groups.append({'params': head_p,   'lr': base_lr})
    if other_p:  groups.append({'params': other_p,  'lr': base_lr * 0.10})

    return optim.AdamW(groups, weight_decay=wd, eps=1e-8)


optimizer = get_optimizer(model)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=8, T_mult=2, eta_min=CFG.MIN_LR)

print("✅ AdamW optimizer with layer-wise LR decay")
print("✅ CosineAnnealingWarmRestarts scheduler")
print("   LR groups:", [f"{g['lr']:.2e}" for g in optimizer.param_groups])


## 🎲 Section 13: MixUp & CutMix (Multi-Label Adapted)

In [ ]:
def mixup_multilabel(x, y_a, y_d, alpha=CFG.MIXUP_ALPHA):
    """
    MixUp for multi-label: linearly blend both images AND labels.
    Both anatomy and disease labels are blended together.
    Zhang et al., ICLR 2018.
    """
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(device)
    return (lam * x + (1-lam) * x[idx],
            lam * y_a + (1-lam) * y_a[idx],
            lam * y_d + (1-lam) * y_d[idx])


def cutmix_multilabel(x, y_a, y_d, alpha=CFG.CUTMIX_ALPHA):
    """
    CutMix for multi-label: patch labels weighted by pixel area ratio.
    Yun et al., ICCV 2019.
    """
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(device)
    W, H = x.size(3), x.size(2)
    cut_w = int(W * np.sqrt(1.0 - lam))
    cut_h = int(H * np.sqrt(1.0 - lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w//2, 0, W)
    x2 = np.clip(cx + cut_w//2, 0, W)
    y1 = np.clip(cy - cut_h//2, 0, H)
    y2 = np.clip(cy + cut_h//2, 0, H)
    mix_x = x.clone()
    mix_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam   = 1 - (x2-x1)*(y2-y1) / (W*H)
    return (mix_x,
            lam * y_a + (1-lam) * y_a[idx],
            lam * y_d + (1-lam) * y_d[idx])

print("✅ MixUp / CutMix (multi-label) ready")


## 🏋️ Section 14: Training Loop

In [ ]:
class EarlyStopping:
    def __init__(self, patience=CFG.PATIENCE, mode='max'):
        self.patience = patience; self.mode = mode
        self.counter  = 0; self.best = None; self.stop = False

    def __call__(self, val):
        if self.best is None:
            self.best = val
        elif (self.mode=='max' and val > self.best + 1e-4) or              (self.mode=='min' and val < self.best - 1e-4):
            self.best = val; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def train_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0

    loop = tqdm(loader, desc=f'Epoch {epoch:02d} [Train]', leave=False)
    for imgs, anatomy_lbl, disease_lbl in loop:
        imgs        = imgs.to(device, non_blocking=True)
        anatomy_lbl = anatomy_lbl.to(device, non_blocking=True)
        disease_lbl = disease_lbl.to(device, non_blocking=True)

        # ── MixUp / CutMix ────────────────────────────────────
        if random.random() < CFG.MIXUP_PROB:
            if random.random() < 0.5:
                imgs, anatomy_lbl, disease_lbl = mixup_multilabel(
                    imgs, anatomy_lbl, disease_lbl)
            else:
                imgs, anatomy_lbl, disease_lbl = cutmix_multilabel(
                    imgs, anatomy_lbl, disease_lbl)

        # ── Forward + loss ────────────────────────────────────
        with torch.cuda.amp.autocast():
            a_logits, d_logits = model(imgs)
            a_loss = anatomy_criterion(a_logits, anatomy_lbl)
            d_loss = disease_criterion(d_logits, disease_lbl)
            loss   = a_loss + CFG.AUX_WEIGHT * d_loss

        # ── Backward ──────────────────────────────────────────
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=f'{loss.item():.4f}',
                         lr=f'{optimizer.param_groups[-1]["lr"]:.2e}')

    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(model, loader):
    model.eval()
    all_probs  = []
    all_labels = []
    total_loss = 0.0

    for imgs, anatomy_lbl, disease_lbl in tqdm(loader, desc='[Val]', leave=False):
        imgs        = imgs.to(device, non_blocking=True)
        anatomy_lbl = anatomy_lbl.to(device, non_blocking=True)
        disease_lbl = disease_lbl.to(device, non_blocking=True)

        with torch.cuda.amp.autocast():
            a_logits, d_logits = model(imgs)
            a_loss = anatomy_criterion(a_logits, anatomy_lbl)
            d_loss = disease_criterion(d_logits, disease_lbl)
            loss   = a_loss + CFG.AUX_WEIGHT * d_loss

        total_loss += loss.item()
        all_probs.append(torch.sigmoid(a_logits).cpu().numpy())
        all_labels.append(anatomy_lbl.cpu().numpy())

    all_probs  = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)
    binary     = (all_probs > CFG.DEFAULT_THRESHOLD).astype(int)
    f1  = f1_score(all_labels, binary, average='macro', zero_division=0)
    return total_loss / len(loader), f1, all_probs, all_labels

print("✅ Training functions ready")


In [ ]:
# ══════════════════════════════════════════════════════════════
# MAIN TRAINING LOOP
# ══════════════════════════════════════════════════════════════
scaler     = torch.cuda.amp.GradScaler()
early_stop = EarlyStopping(patience=CFG.PATIENCE, mode='max')
history    = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'lr': []}
BEST_F1    = 0.0

print("\n" + "═"*65)
print("  🚀  ANATOMY DETECTION — TRAINING STARTED")
print("═"*65 + "\n")

for epoch in range(1, CFG.EPOCHS + 1):
    tr_loss            = train_epoch(model, train_loader, optimizer, scaler, epoch)
    val_loss, val_f1, _, _ = val_epoch(model, val_loader)
    scheduler.step()
    lr = optimizer.param_groups[-1]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['lr'].append(lr)

    flag = ''
    if val_f1 > BEST_F1:
        BEST_F1 = val_f1
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_f1': val_f1, 'anatomy_cols': CFG.ANATOMY_COLS},
                   CFG.MODEL_PATH)
        flag = '  ← 🔥 best'

    print(f"Epoch [{epoch:02d}/{CFG.EPOCHS}]  "
          f"TrLoss={tr_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"ValF1={val_f1:.4f}  LR={lr:.2e}{flag}")

    if early_stop(val_f1):
        print(f"\n⛔ Early stopping at epoch {epoch}")
        break

print(f"\n{'═'*65}")
print(f"  🏆  DONE  |  Best Val F1 = {BEST_F1:.4f}")
print("═"*65)


## 📈 Section 15: Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep, history['train_loss'], 'b-o', ms=4, label='Train')
axes[0].plot(ep, history['val_loss'],   'r-o', ms=4, label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['val_f1'], 'g-o', ms=4)
axes[1].axhline(BEST_F1, ls='--', color='navy', label=f'Best={BEST_F1:.4f}')
axes[1].set_title('Val Macro F1', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history['lr'], 'purple', ls='--', marker='.')
axes[2].set_title('Learning Rate', fontweight='bold')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/training_curves.png', dpi=150)
plt.show()


## 🔍 Section 16: TTA Inference on Test Set

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────
ckpt = torch.load(CFG.MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
print(f"✅ Loaded epoch {ckpt['epoch']}  |  Val F1 = {ckpt['val_f1']:.4f}")


@torch.no_grad()
def predict_tta(model, df, batch_size=CFG.BATCH_SIZE):
    """
    5-view TTA inference.
    Each image is passed through 5 augmented versions;
    sigmoid probabilities are averaged before thresholding.
    """
    model.eval()
    all_run_probs = []

    for tf in get_tta_transforms():
        ds     = GalarDataset(df, transform=tf)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=CFG.NUM_WORKERS)
        run_probs = []
        for imgs, _, _ in tqdm(loader, desc='TTA', leave=False):
            imgs = imgs.to(device)
            with torch.cuda.amp.autocast():
                probs = model.predict_proba(imgs)
            run_probs.append(probs.cpu().numpy())
        all_run_probs.append(np.vstack(run_probs))

    avg_probs  = np.mean(all_run_probs, axis=0)          # (N, 10)
    true_labels = df[CFG.ANATOMY_COLS].values.astype(float)
    preds_05   = (avg_probs > CFG.DEFAULT_THRESHOLD).astype(int)
    return preds_05, avg_probs, true_labels


print("\n🔮 Running TTA inference on test set...")
test_preds, test_probs, test_labels = predict_tta(model, test_df)

f1_05 = f1_score(test_labels, test_preds, average='macro', zero_division=0)
print(f"\n✅ Macro F1 @ threshold=0.50 : {f1_05:.4f}")


## 🎯 Section 17: Per-Class Threshold Optimization

In [ ]:
print("🔍 Searching optimal threshold per anatomy class...\n")

best_thresholds = {}
best_f1s        = {}

for i, cls in enumerate(CFG.ANATOMY_COLS):
    best_t, best_f1 = CFG.DEFAULT_THRESHOLD, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        p  = (test_probs[:, i] > t).astype(int)
        f1 = f1_score(test_labels[:, i], p, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresholds[cls] = float(round(best_t, 2))
    best_f1s[cls]        = round(best_f1, 4)
    print(f"  {cls:25s}  t={best_t:.2f}  F1={best_f1:.4f}")

# Apply optimized thresholds
opt_preds = np.stack([
    (test_probs[:, i] > best_thresholds[cls]).astype(int)
    for i, cls in enumerate(CFG.ANATOMY_COLS)
], axis=1)

f1_opt = f1_score(test_labels, opt_preds, average='macro', zero_division=0)
print(f"\n{'='*55}")
print(f"  Macro F1 — global threshold 0.50  : {f1_05:.4f}")
print(f"  Macro F1 — optimized thresholds   : {f1_opt:.4f}  (+{f1_opt-f1_05:.4f})")
print(f"{'='*55}")

json.dump(best_thresholds,
          open(f'{CFG.OUTPUT_DIR}/best_thresholds.json', 'w'), indent=2)
print("✅ Thresholds saved")


## 📊 Section 18: Full Evaluation Report

In [ ]:
print("\n" + "═"*60)
print("  📋  CLASSIFICATION REPORT — ANATOMY DETECTION (TTA)")
print("═"*60)
print(classification_report(
    test_labels, opt_preds,
    target_names=CFG.ANATOMY_COLS,
    zero_division=0, digits=4
))


In [ ]:
# ── Per-class F1 bar chart ────────────────────────────────────
per_f1 = f1_score(test_labels, opt_preds, average=None, zero_division=0)
colors = ['#2ecc71' if v >= 0.8 else '#f39c12' if v >= 0.5 else '#e74c3c'
          for v in per_f1]

plt.figure(figsize=(13, 5))
bars = plt.bar(CFG.ANATOMY_COLS, per_f1, color=colors, edgecolor='k', lw=0.5)
for b, v in zip(bars, per_f1):
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
             f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.axhline(f1_opt, ls='--', color='navy', label=f'Macro F1={f1_opt:.4f}')
plt.ylim(0, 1.15); plt.xticks(rotation=35, ha='right')
plt.title('Per-Class F1 — Anatomy Region Detection', fontsize=13, fontweight='bold')
plt.ylabel('F1 Score'); plt.legend(); plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/per_class_f1.png', dpi=150)
plt.show()


In [ ]:
# ── Per-class confusion matrices (2×2) ───────────────────────
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for i, cls in enumerate(CFG.ANATOMY_COLS):
    cm = confusion_matrix(test_labels[:, i], opt_preds[:, i])
    # Pad to 2×2 if only one class present in test
    if cm.shape == (1, 1):
        pad = np.zeros((2, 2), dtype=int); pad[0,0] = cm[0,0]; cm = pad

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Pred 0','Pred 1'],
                yticklabels=['True 0','True 1'], linewidths=0.5)
    axes[i].set_title(f'{cls}\nF1={per_f1[i]:.3f}', fontsize=10, fontweight='bold')

plt.suptitle('Per-Class Confusion Matrices — Anatomy Detection',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/confusion_matrices.png', dpi=150)
plt.show()


In [ ]:
# ── ROC Curves ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
auc_scores = []

for i, cls in enumerate(CFG.ANATOMY_COLS):
    if test_labels[:, i].sum() == 0:
        print(f"  ⚠️  {cls} — no positive samples in test, skipping AUC")
        continue
    fpr, tpr, _ = roc_curve(test_labels[:, i], test_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    auc_scores.append(roc_auc)
    ax.plot(fpr, tpr, lw=1.8, label=f'{cls}  (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Random (AUC=0.500)')
ax.set_title('ROC Curves — Anatomy Detection (TTA)', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/roc_curves.png', dpi=150)
plt.show()

mean_auc = float(np.mean(auc_scores)) if auc_scores else 0.0
print(f"\nMean AUC : {mean_auc:.4f}")


## 🔬 Section 19: GradCAM++ Explainability

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# Wrapper: GradCAM needs a model that returns a single tensor
class AnatomyOnlyWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x)[0]   # anatomy logits only

wrapped      = AnatomyOnlyWrapper(model)
target_layer = model.backbone.blocks[-2]   # second-to-last block → best spatial resolution
cam          = GradCAMPlusPlus(model=wrapped, target_layers=[target_layer])

tf_val = get_val_transforms()

def show_gradcam(df, n=6, title="GradCAM++ — Model Attention Maps"):
    indices = random.sample(range(len(df)), min(n, len(df)))
    fig = plt.figure(figsize=(15, n * 4))

    for ri, idx in enumerate(indices):
        row      = df.iloc[idx]
        orig_img = np.array(Image.open(row['img_path']).convert('RGB')
                            .resize((CFG.IMG_SIZE, CFG.IMG_SIZE)))

        true_regions = [c for c in CFG.ANATOMY_COLS if row[c] == 1]

        inp = tf_val(image=orig_img)['image'].unsqueeze(0).to(device)
        inp.requires_grad = True

        heatmap = cam(input_tensor=inp)[0]
        overlay = show_cam_on_image(orig_img.astype(np.float32)/255., heatmap, use_rgb=True)

        # Model prediction
        model.eval()
        with torch.no_grad():
            probs = model.predict_proba(inp)[0].cpu().numpy()
        pred_regions = [CFG.ANATOMY_COLS[j] for j in range(CFG.NUM_ANATOMY)
                        if probs[j] > best_thresholds[CFG.ANATOMY_COLS[j]]]

        ax1 = fig.add_subplot(n, 3, ri*3+1)
        ax2 = fig.add_subplot(n, 3, ri*3+2)
        ax3 = fig.add_subplot(n, 3, ri*3+3)

        ax1.imshow(orig_img); ax1.axis('off')
        ax1.set_title(f'True: {true_regions}', fontsize=8)

        ax2.imshow(heatmap, cmap='jet'); ax2.axis('off')
        ax2.set_title('GradCAM++ Heatmap', fontsize=9)

        match = '✅' if set(pred_regions)==set(true_regions) else '⚠️'
        ax3.imshow(overlay); ax3.axis('off')
        ax3.set_title(f'{match} Pred: {pred_regions}', fontsize=8)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{CFG.OUTPUT_DIR}/gradcam.png', dpi=150)
    plt.show()

show_gradcam(test_df, n=6)


## 🎥 Section 20: Patient Video Timeline + Temporal Smoothing

In [ ]:
def predict_patient_video(model, patient_id, smooth_window=CFG.TEMPORAL_WINDOW):
    """
    Runs frame-by-frame inference on a full patient video.

    Temporal smoothing: applies a rolling-average over consecutive
    frames to reduce single-frame prediction noise — important
    because capsule endoscopy produces continuous video where
    adjacent frames should have similar anatomy labels.

    Returns a DataFrame with one row per frame containing:
      - raw probabilities per anatomy class
      - smoothed probabilities
      - binary predictions using optimized thresholds
    """
    model.eval()
    pid_str    = str(patient_id)
    patient_df = master_df[master_df['patient_id'] == pid_str].sort_values('frame').reset_index(drop=True)
    print(f"Patient {patient_id}: {len(patient_df):,} frames")

    tf         = get_val_transforms()
    raw_probs  = []

    for _, row in tqdm(patient_df.iterrows(), total=len(patient_df),
                       desc=f'Patient {patient_id}'):
        try:
            img  = np.array(Image.open(row['img_path']).convert('RGB'))
            inp  = tf(image=img)['image'].unsqueeze(0).to(device)
            with torch.no_grad():
                p = model.predict_proba(inp)[0].cpu().numpy()
            raw_probs.append(p)
        except Exception:
            raw_probs.append(np.zeros(CFG.NUM_ANATOMY))

    raw_arr  = np.array(raw_probs)   # (N_frames, 10)

    # Rolling average smoothing
    smooth_df = (pd.DataFrame(raw_arr, columns=CFG.ANATOMY_COLS)
                   .rolling(window=smooth_window, center=True, min_periods=1)
                   .mean())

    # Binary predictions with per-class optimal thresholds
    result = pd.DataFrame({
        'frame':    patient_df['frame'].values,
        'img_path': patient_df['img_path'].values,
    })
    for col in CFG.ANATOMY_COLS:
        result[f'{col}_raw']    = raw_arr[:, CFG.ANATOMY_COLS.index(col)]
        result[f'{col}_smooth'] = smooth_df[col].values
        result[f'{col}_pred']   = (smooth_df[col].values > best_thresholds[col]).astype(int)

    return result


# ── Run on first test patient ─────────────────────────────────
pid      = test_pids[0]
video_df = predict_patient_video(model, pid)


In [ ]:
# ── Anatomy timeline plot ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(22, 10), sharex=True)

colors = sns.color_palette('tab10', CFG.NUM_ANATOMY)
frames = video_df['frame'].values

# Top: smoothed probabilities
for i, col in enumerate(CFG.ANATOMY_COLS):
    axes[0].plot(frames, video_df[f'{col}_smooth'], lw=1.8,
                 color=colors[i], label=col, alpha=0.9)
    axes[0].axhline(best_thresholds[col], color=colors[i],
                    ls=':', lw=0.7, alpha=0.4)
axes[0].set_title(f'Patient {pid} — Smoothed Anatomy Probabilities',
                  fontweight='bold', fontsize=13)
axes[0].set_ylabel('Probability'); axes[0].set_ylim(-0.05, 1.1)
axes[0].legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
axes[0].grid(alpha=0.3)

# Bottom: binary detection heatmap
binary_matrix = np.array([
    video_df[f'{col}_pred'].values for col in CFG.ANATOMY_COLS
])
axes[1].imshow(binary_matrix, aspect='auto', cmap='Blues',
               extent=[frames[0], frames[-1], -0.5, CFG.NUM_ANATOMY - 0.5])
axes[1].set_yticks(range(CFG.NUM_ANATOMY))
axes[1].set_yticklabels(CFG.ANATOMY_COLS, fontsize=9)
axes[1].set_xlabel('Frame Number', fontsize=11)
axes[1].set_title('Binary Detections (blue = detected)', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/patient_timeline.png', dpi=150)
plt.show()


In [ ]:
# ── Patient anatomy summary report ───────────────────────────
print(f"\n{'='*60}")
print(f"  📋  PATIENT {pid} — ANATOMY DETECTION REPORT")
print(f"{'='*60}")
print(f"  {'Region':25s}  {'Status':12s}  {'Frames':>8}  {'Coverage':>9}  {'Max Prob':>9}")
print(f"  {'-'*58}")

for col in CFG.ANATOMY_COLS:
    detected = video_df[f'{col}_pred'].sum()
    total    = len(video_df)
    pct      = detected / total * 100
    max_p    = video_df[f'{col}_smooth'].max()
    status   = '✅ DETECTED' if detected > 0 else '➖ not seen'
    print(f"  {col:25s}  {status:12s}  {detected:>8,}  {pct:>8.1f}%  {max_p:>9.3f}")

print(f"{'='*60}")


## 💾 Section 21: Single-Frame Inference Pipeline

In [ ]:
# Save model artifacts
model_info = {
    'model_name':      CFG.MODEL_NAME,
    'anatomy_cols':    CFG.ANATOMY_COLS,
    'img_size':        CFG.IMG_SIZE,
    'best_thresholds': best_thresholds,
    'test_macro_f1':   float(f1_opt),
    'mean_auc':        float(mean_auc),
}
json.dump(model_info,
          open(f'{CFG.OUTPUT_DIR}/model_info.json','w'), indent=2)
print("✅ Model info saved")


def predict_frame(image_path, model, thresholds=None, use_tta=True):
    """
    Full inference on a single frame image.

    Args:
        image_path : path to PNG/JPG file
        model      : trained AnatomyDetector
        thresholds : dict of per-class thresholds (default: best_thresholds)
        use_tta    : whether to apply 5-view TTA

    Returns:
        detections : dict {region_name: probability} for detected regions
    """
    if thresholds is None:
        thresholds = best_thresholds

    model.eval()
    img = np.array(Image.open(image_path).convert('RGB'))

    if use_tta:
        probs_list = []
        for tf in get_tta_transforms():
            inp  = tf(image=img)['image'].unsqueeze(0).to(device)
            with torch.no_grad():
                probs_list.append(model.predict_proba(inp)[0].cpu().numpy())
        probs = np.mean(probs_list, axis=0)
    else:
        tf   = get_val_transforms()
        inp  = tf(image=img)['image'].unsqueeze(0).to(device)
        with torch.no_grad():
            probs = model.predict_proba(inp)[0].cpu().numpy()

    detections = {
        col: float(probs[i])
        for i, col in enumerate(CFG.ANATOMY_COLS)
        if probs[i] > thresholds[col]
    }

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].imshow(Image.open(image_path).convert('RGB').resize((CFG.IMG_SIZE, CFG.IMG_SIZE)))
    axes[0].set_title('Input Frame', fontweight='bold'); axes[0].axis('off')

    bar_colors = ['#2ecc71' if probs[i] > thresholds[CFG.ANATOMY_COLS[i]]
                  else '#cccccc' for i in range(CFG.NUM_ANATOMY)]
    axes[1].barh(CFG.ANATOMY_COLS, probs, color=bar_colors, edgecolor='k', lw=0.4)
    for i, (col, p) in enumerate(zip(CFG.ANATOMY_COLS, probs)):
        axes[1].axvline(x=thresholds[col], color='red', ls='--', alpha=0.3, lw=0.8)
    axes[1].set_xlabel('Probability'); axes[1].set_xlim(0, 1)
    axes[1].set_title('Anatomy Probabilities (green = detected)', fontweight='bold')
    plt.tight_layout()
    plt.show()

    return detections


# ── Demo on a random test frame ───────────────────────────────
sample_path = test_df.sample(1, random_state=SEED).iloc[0]['img_path']
print(f"\nRunning inference on: {sample_path}")
results = predict_frame(sample_path, model, use_tta=True)
print("\n📍 Detected anatomy regions:")
if results:
    for region, prob in sorted(results.items(), key=lambda x: -x[1]):
        print(f"  {region:25s}  {prob:.3f}")
else:
    print("  (no regions detected above threshold)")


## 📊 Section 22: Final Results Summary

| Metric | Value |
|--------|-------|
| Test Macro F1 (threshold=0.5) | see cell output |
| Test Macro F1 (optimized thresholds) | see cell output |
| Mean AUC | see cell output |

---

### Architecture Decisions

| Component | Choice | Why |
|-----------|--------|-----|
| Backbone | EfficientNet-B4 NoisyStudent | Pretrained on 300M images — richest features available |
| Pooling | GeM | Learnable p parameter adapts to medical imaging statistics |
| Heads | Shared trunk + separate heads | Anatomy and disease co-occurrence improves shared features |
| Loss | Focal + Label Smoothing + pos_weight | Three-way imbalance handling |
| Sampler | WeightedRandomSampler | Rare regions (ampulla, mouth) seen as often as stomach |
| Aug | Albumentations + MixUp + CutMix | Simulate VCE variation; reduce overfitting |
| Split | Patient-level | Prevents patient-specific leakage |
| LR | Layer-wise decay + CosineWarmRestarts | Preserve pretrained features; escape local minima |
| Inference | 5-view TTA + temporal smoothing | +1-3% F1; reduce video prediction flicker |
| Explainability | GradCAM++ on blocks[-2] | Sharper spatial maps than last layer |

### References
- Tan & Le (2019). EfficientNet. ICML.
- Lin et al. (2017). Focal Loss / RetinaNet. ICCV.
- Zhang et al. (2018). MixUp. ICLR.
- Yun et al. (2019). CutMix. ICCV.
- Chattopadhyay et al. (2018). GradCAM++. WACV.
- Radenovic et al. (2019). GeM Pooling. TPAMI.
- Pogorelov et al. (2017). KVASIR Dataset. ACM MMSys.
